## Load and plot collinear up/down Wannier90 TB bands with PythTB

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pythtb import w90  # PythTB's Wannier90 interface

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pythtb import w90

# --------------------------
# USER SETTINGS
# --------------------------
seed_up = "wannier90.1"
seed_dn = "wannier90.2"

w90_dir = "/Users/guymoore/Documents/ResearchProjects/magnetism/BiFeO3_TB/00_R3c_cl"

n_k_per_segment = 60
fermi_level = 4.50820836  # set if you know EF (eV)

G = np.array([0.0, 0.0, 0.0])
k_delta = 0.1
S1  = np.array([-k_delta, 0.5,  k_delta])
S2  = np.array([ k_delta, 0.5, -k_delta])

# S1  = np.array([ 0.5, 0.5,  0.0])
# S2  = np.array([ 0.5, 0.5,  0.0])

k_nodes = [
    ("S1", S1),
    ("Γ", G),
    ("S2", S2),
]

# --------------------------
# Helper: build a k-path
# --------------------------
def build_kpath(k_nodes, n_per_segment):
    labels = [k[0] for k in k_nodes]
    kpts = np.array([k[1] for k in k_nodes], dtype=float)

    k_list = []
    x_list = []
    tick_positions = [0.0]
    x = 0.0

    for i in range(len(kpts) - 1):
        k0 = kpts[i]
        k1 = kpts[i + 1]
        for j in range(n_per_segment):
            t = j / float(n_per_segment)
            k = (1 - t) * k0 + t * k1
            k_list.append(k)
            x_list.append(x)
            if j < n_per_segment - 1:
                dk = np.linalg.norm((k1 - k0) / n_per_segment)
                x += dk
        tick_positions.append(x)

    k_list.append(kpts[-1])
    x_list.append(x)

    return np.array(k_list), np.array(x_list), labels, np.array(tick_positions)

k_vec, x_vec, labels, tick_pos = build_kpath(k_nodes, n_k_per_segment)

# --------------------------
# Load Wannier90 + build PythTB models
# --------------------------
# Doc-style: w90(folder, seedname)
w90_up = w90(w90_dir, seed_up)
w90_dn = w90(w90_dir, seed_dn)

print("Creating TB model...")

# Convert to PythTB tight-binding models
# min_hopping_norm prunes tiny hoppings; set to 0.0 to keep everything
tb_up = w90_up.model(min_hopping_norm=0.01)
tb_dn = w90_dn.model(min_hopping_norm=0.01)

print("Solving TB model...")

# --------------------------
# Compute bands using PythTB
# --------------------------
# solve_all expects k-points in fractional reciprocal coordinates
E_up = tb_up.solve_all(k_vec) - fermi_level   # shape: (n_bands, n_k)
E_dn = tb_dn.solve_all(k_vec) - fermi_level

print("Plotting...")

# --------------------------
# Plot
# --------------------------
plt.figure(dpi=300, figsize=(4.5, 4.5))

for n in range(E_up.shape[0]):2
    plt.plot(x_vec, E_up[n, :], linewidth=0.9, alpha=0.9, color="b", linestyle="-")

for n in range(E_dn.shape[0]):
    plt.plot(x_vec, E_dn[n, :], linewidth=0.9, alpha=0.9, color="r", linestyle="--")

for xp in tick_pos:
    plt.axvline(xp, linewidth=0.8)

plt.xticks(tick_pos, labels)
plt.ylabel(r"Energy $E - E_F$ (eV)")
plt.xlabel("k-path")
plt.title("Wannier90 TB bands: spin-up (solid) vs spin-down (dashed)")
# plt.ylim(-3, 3)
plt.tight_layout()
plt.show()
